# GPT5-mini text generation (rewriting GPT5-mini answers)

### Import libraries

In [1]:
import numpy as np
import pandas as pd

import os
import json

from google.colab import drive, userdata
from openai import OpenAI

### Set up Google Drive mounting and define input/output paths:

In [2]:
# Mount the google drive folder
drive.mount('/content/drive', force_remount = True) # force reconnection

# Define directory, input, and output paths
project_directory = "/content/drive/MyDrive/Colab Notebooks/DS266/final_project"
input_file = os.path.join(project_directory, "data/everything_but_reddit_unstandardized_with_kimi.parquet")
output_file = os.path.join(project_directory, "data/gpt data/gpt5mini_gpt.jsonl")

# Ensure the output directory exists
if not os.path.exists(project_directory):
  os.makedirs(project_directory)

# Get access to the seceret api key
os.environ["OPENAI_API_KEY"] = userdata.get('HC3_GPT5mini')
client = OpenAI()

Mounted at /content/drive


### Read the input file:

In [3]:
# Parquet file
df = pd.read_parquet(input_file)
print('Sucessfully read the input file with the questions and original answers!')

print(f'The input file contains {len(df)} records/rows.')
print(f'Column names: {list(df.columns)}')

print('\nThe first 5 rows:')
display(df.head())

Sucessfully read the input file with the questions and original answers!
The input file contains 5306 records/rows.
Column names: ['question', 'human_answers', 'source', 'gemini_answers', 'gpt5mini_answer', 'claude_haiku_4.5', 'kimi_k2_answers']

The first 5 rows:


,question,human_answers,source,gemini_answers,gpt5mini_answer,claude_haiku_4.5,kimi_k2_answers
0,what composer used sound mass,"['Composers and works include Barbara Kolb , P...",open_qa,The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"The term **""sound mass""** (or **""sound-mass co..."
1,where did the persian war take place,['The Greco-Persian Wars (also often called th...,open_qa,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The **Persian Wars** (also called the **Greco-...
2,what are add ons,"['Plug-in (computing) , a piece of software wh...",open_qa,An **add-on** is something extra or additional...,"""Add-ons"" (sometimes written ""add ons"", ""addon...",# Add-ons\n\nAdd-ons are **additional features...,**Add-ons** are extra software components or e...
3,how does a dredge work?,['Dredging is an excavation activity or operat...,open_qa,A dredge is essentially a floating machine des...,"A dredge is a machine used to remove sediment,...",# How a Dredge Works\n\nA dredge is a vessel o...,A **dredge** is a machine or system designed t...
4,what classes are considered humanities,['The humanities are academic disciplines that...,open_qa,The humanities are academic disciplines that s...,Humanities are academic disciplines that study...,# Humanities Classes\n\nHumanities typically i...,“Humanities” is the broad family of discipline...


### Print the sample question and LLM answer:

In [4]:
print('Sample question:')
display(df['question'][0])

print('\nSample GPT original answer:')
display(df['gpt5mini_answer'][0])

Sample question:


'what composer used sound mass'


Sample GPT original answer:


'Several 20th-century composers used the "sound mass" technique (treating clusters, textures, and timbre as the primary musical material rather than melody/harmony). Notable examples:\n\n- György Ligeti — Atmosphères (1961), Lux Aeterna (1966) — dense micropolyphonic textures.\n- Krzysztof Penderecki — Threnody to the Victims of Hiroshima (1960) — extreme clusters and extended string effects.\n- Iannis Xenakis — Metastasis (1954–55), Pithoprakta (1955–56) — stochastic clouds and textures.\n- Edgard Varèse — Ionisation (1931), Poème électronique (1958) — emphasis on timbre and massed sonorities.\n- Karlheinz Stockhausen and Luigi Nono — various works exploring timbral and textural blocks.\n\nIf you want, I can give short audio/score examples or explain how the technique works in one of these pieces.'

### Rewrite GPT5-mini answers in the new column `gpt5mini_gpt`:

In [5]:
# Look for where the generation was left off

last_row_id = -1

if os.path.exists(output_file):
  output_df = pd.read_json(output_file, lines = True) # JSONL
  last_row_id = output_df.index[-1]

print('No output file exists.' if last_row_id < 0 else f'Last row generated (row id): {last_row_id}')

Last row generated (row id): 5305


In [6]:
# Start appending to the file from where it's left off

with open(output_file, 'a') as f:

  # Keep track no-answer rows
  no_answer = 0

  # Iterate through each record/row
  for i, row in df.iterrows():

    # Print that the generation is already done if no rows are left to be generated
    if i == len(df) - 1:
      print(f'GPT5-mini {len(df)} answer generations already completed!')

    # Continue if current row is less than or equal to last_row_id
    if i <= last_row_id:
      continue

    # Convert each row to dictionary becasue the output file is going to be json
    row = row.to_dict()

    # Get the original answer to be rewritten
    original = row['gpt5mini_answer']

    # Generate response from GPT-5mini
    try:
      gpt5mini_response = client.chat.completions.create(model = 'gpt-5-mini',
                                                         messages = [{'role': 'user', 'content': f"Refine the following text:\n\n{original}\n\nOutput only the rewritten text, nothing else."}],
                                                         max_completion_tokens = 1024,
                                                         reasoning_effort = 'low')  # Increase efficiency and comparability between models
      gpt5mini_answer = gpt5mini_response.choices[0].message.content

    except Exception as e:
      print(f'Error when processing row {i + 1} (index + 1) answer: {e}')
      no_answer += 1
      gpt5mini_answer = None

    # Insert the GPT-5mini answers into the dictionary (each row has a new column in the output file)
    row['gpt5mini_gpt'] = gpt5mini_answer

    # Write the row with new column into the output file
    f.write(json.dumps(row) + '\n')   # '\n' so that each row starts with a new line


    # Print out the progress when generating
    if i % 100 == 0:

      # Force writing to the os in case the program crashes
      f.flush()
      os.fsync(f.fileno())

      # Print the progress and saving status
      if i != 0 and (i - last_row_id) > 100:
        print('Finished and saved!')
      print(f'Start generating 100 answers for batch {(i // 100) + 1} ....', end = ' ')

    elif i == len(df) - 1:
      print(f'Finished {len(df)} answer generations!')

GPT5-mini 5306 answer generations already completed!


In [7]:
# Convert JSONL to CSV
out_df = pd.read_json(output_file, lines = True)

output_file_csv = os.path.join(project_directory, 'data/gpt data/gpt5mini_gpt.csv')
out_df.to_csv(output_file_csv, index = False)

### Read the output file after rewriting:

In [8]:
out_df = pd.read_csv(output_file_csv)
print('Sucessfully read the output file with GPT5-mini rewriting answers!')

print(f'The output file contains {len(out_df)} records/rows.')
print(f'New column names: {list(out_df.columns)}')

print('\nThe first 5 rows:')
display(out_df.head())

Sucessfully read the output file with GPT5-mini rewriting answers!
The output file contains 5306 records/rows.
New column names: ['question', 'human_answers', 'source', 'gemini_answers', 'gpt5mini_answer', 'claude_haiku_4.5', 'kimi_k2_answers', 'gpt5mini_gpt']

The first 5 rows:


,question,human_answers,source,gemini_answers,gpt5mini_answer,claude_haiku_4.5,kimi_k2_answers,gpt5mini_gpt
0,what composer used sound mass,"['Composers and works include Barbara Kolb , P...",open_qa,The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"The term **""sound mass""** (or **""sound-mass co...","Several 20th-century composers employed the ""s..."
1,where did the persian war take place,['The Greco-Persian Wars (also often called th...,open_qa,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The **Persian Wars** (also called the **Greco-...,If you mean the Greco–Persian Wars (commonly c...
2,what are add ons,"['Plug-in (computing) , a piece of software wh...",open_qa,An **add-on** is something extra or additional...,"""Add-ons"" (sometimes written ""add ons"", ""addon...",# Add-ons\n\nAdd-ons are **additional features...,**Add-ons** are extra software components or e...,"""Add-ons"" (also written add-ons, add ons, addo..."
3,how does a dredge work?,['Dredging is an excavation activity or operat...,open_qa,A dredge is essentially a floating machine des...,"A dredge is a machine used to remove sediment,...",# How a Dredge Works\n\nA dredge is a vessel o...,A **dredge** is a machine or system designed t...,"A dredge is a machine used to remove sediment,..."
4,what classes are considered humanities,['The humanities are academic disciplines that...,open_qa,The humanities are academic disciplines that s...,Humanities are academic disciplines that study...,# Humanities Classes\n\nHumanities typically i...,“Humanities” is the broad family of discipline...,Humanities are academic disciplines that exami...


### Print the sample question and GPT5-mini answers:

In [9]:
print('Sample question:')
display(out_df['question'].iloc[-1])

print('\nSample GPT original answer:')
display(out_df['gpt5mini_answer'].iloc[-1])
print('\nSample GPT answer rewritten by GPT5-mini:')
display(out_df['gpt5mini_gpt'].iloc[-1])

Sample question:


'Are BP of 119/65 and pulse of 35 causes for concern?Yesterday I was very dizzy, short of breath, heart rate around 35,BP 119/65. I am 51,female, overweight and have RA and Fibermyalgia. Do you know what might have caused this episode yesterday? I figure I would not feel okay today if it had been a heart attack.'


Sample GPT original answer:


'A resting heart rate around 35 beats per minute is lower than normal for an adult and — especially when it’s accompanied by dizziness and shortness of breath — is a sign that needs urgent medical attention.\n\nWhy this is concerning\n- Normal resting heart rate for most adults is about 60–100 bpm. A sustained rate of 35 bpm (bradycardia) can reduce cardiac output and cause symptoms such as lightheadedness, syncope (fainting), shortness of breath, and fatigue.  \n- Your blood pressure of 119/65 is within a normal range, but a normal BP does not rule out a dangerous problem if the heart rate is very low and you are symptomatic.\n\nPossible causes (not a diagnosis)\nSeveral things can cause symptomatic bradycardia and the episode you describe:\n- Medications (common causes): beta‑blockers, nondihydropyridine calcium‑channel blockers (verapamil, diltiazem), digoxin, certain antiarrhythmics, some antidepressants or opioids. Review any prescription, over‑the‑counter, or herbal meds you take


Sample GPT answer rewritten by GPT5-mini:


'A resting heart rate around 35 beats per minute is low for an adult and — especially when accompanied by dizziness and shortness of breath — requires urgent medical attention.\n\nWhy this is concerning\n- Normal resting heart rate for most adults is about 60–100 bpm. A sustained rate near 35 bpm (bradycardia) can reduce cardiac output and cause lightheadedness, syncope (fainting), shortness of breath, and fatigue.  \n- Your blood pressure of 119/65 is normal, but normal BP does not rule out a serious problem when the heart rate is very low and you are symptomatic.\n\nPossible causes (not a diagnosis)\nCommon causes of symptomatic bradycardia include:\n- Medications: beta‑blockers, nondihydropyridine calcium‑channel blockers (verapamil, diltiazem), digoxin, some antiarrhythmics, certain antidepressants, and opioids. Review all prescription, over‑the‑counter, and herbal medicines.  \n- Cardiac conduction disease: sinus node dysfunction (sick sinus), atrioventricular block, or other rhyt

### Check NaN counts

In [10]:
pd.isna(out_df['gpt5mini_gpt']).value_counts()

,count
gpt5mini_gpt,
False,5302
True,4
